# 조선시대 신하 말투 Fine-tuning
**모델:** Qwen2.5-1.5B-Instruct  
**방법:** QLoRA (Unsloth)  
**환경:** Google Colab (무료 티어)

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [ ]:
!pip install -q --upgrade datasets
!pip install -q unsloth

## 2. 모델 로드
Unsloth로 4bit 양자화된 모델을 불러옵니다.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)
print("✅ 모델 로드 완료")

## 3. LoRA 설정
학습할 레이어만 선택해서 메모리를 아낍니다.

### 각 파라미터 설명

#### `r = 16` (LoRA rank)
LoRA는 거대한 가중치 행렬 W를 직접 수정하지 않고, **작은 두 행렬 A, B의 곱(A×B)으로 변화량만 학습**합니다.

원래 행렬 W가 `4096 × 4096` 크기라면:
- 직접 학습: **16,777,216개** 파라미터
- LoRA (r=16): `4096×16` + `16×4096` = **131,072개** → 약 **128배 절약**

| r 값 | 용도 |
|------|------|
| 8 | 가벼운 학습, 간단한 스타일 변환 |
| 16 | 가장 흔한 선택, 대부분의 fine-tuning에 적합 |
| 32~64 | 복잡한 태스크, 더 많은 지식 주입 |

#### `lora_alpha = 16` (스케일링 계수)
LoRA가 학습한 변화량을 원래 모델에 **얼마나 세게 반영할지** 조절합니다.

실제 반영 비율 = `lora_alpha / r` = `16 / 16` = **1.0** (가장 일반적인 설정)

#### `lora_dropout = 0`
학습 중 LoRA 레이어의 출력을 랜덤하게 꺼서 과적합을 방지하는 기법입니다. Unsloth에서는 최적화된 커널을 사용하기 때문에 **0으로 두는 것이 권장**됩니다.

#### `bias = "none"`
모델의 bias 파라미터를 학습하지 않습니다. 대부분의 경우 bias를 학습하지 않아도 성능 차이가 거의 없어서 `"none"`이 표준입니다.

#### `use_gradient_checkpointing = "unsloth"`
역전파 시 중간 계산 결과를 전부 저장하지 않고 필요할 때 **다시 계산**해서 메모리를 약 **60~70% 절약**합니다. 속도는 약 20~30% 느려지지만, Colab 무료 T4 GPU(16GB)에서는 사실상 필수입니다. `"unsloth"` 값은 Unsloth 고유의 최적화 버전입니다.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("✅ LoRA 설정 완료")

## 4. 데이터 준비
`joseon_finetune_200.json`을 업로드한 뒤 실행하세요.

In [ ]:
# 파일 업로드
from google.colab import files
uploaded = files.upload()  # joseon_finetune_200.json 선택

In [ ]:
import json
from datasets import Dataset

with open("joseon_finetune_200.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

# Qwen 채팅 템플릿 적용
def format_example(example):
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(raw).map(format_example)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개")
print("\n--- 샘플 확인 ---")
print(dataset[0]["text"])

## 5. 학습 실행
약 5~10분 소요됩니다.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 1024,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        fp16 = True,
        logging_steps = 10,
        output_dir = "./joseon_output",
        report_to = "none",
    ),
)

trainer.train()
print("✅ 학습 완료")

## 6. 결과 비교
fine-tuning 전/후 말투를 비교합니다.

In [ ]:
def generate(prompt, model, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# 테스트할 질문
test_questions = [
    "오늘 점심 뭐 먹을까?",
    "요즘 너무 피곤해.",
    "주말에 어디 가면 좋을까?",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

## 7. 모델 저장 (선택)
LoRA 가중치만 저장합니다 (용량 작음).

In [ ]:
model.save_pretrained("joseon_lora")
tokenizer.save_pretrained("joseon_lora")

# 구글 드라이브에 저장하려면
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained("/content/drive/MyDrive/joseon_lora")

print("✅ 저장 완료 → joseon_lora/")

## 8. 모델 불러오기

모델을 불러온 뒤, LoRA 가중치를 따로 불러와서 적용합니다.

In [ ]:
from unsloth import FastLanguageModel

# 1) 베이스 모델 로드 (학습할 때와 동일한 설정)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

# 2) 저장해둔 LoRA 가중치 얹기
from peft import PeftModel
model = PeftModel.from_pretrained(model, "joseon_lora")